### Prepapra a base para o GLM para a analise da importancia das palavras chaves SLR2

Vamos olhar o quanto as palavras chaves impactam no processo de seleção

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import openai
import numpy as np
import time
import os
from dotenv import load_dotenv, find_dotenv
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
### style de execução
tqdm.pandas()

In [3]:
# 1. Configurar para exibir todas as linhas
pd.set_option('display.max_rows', None)

# 2. Configurar para exibir todas as colunas
pd.set_option('display.max_columns', None)

## Leitura e processamento dos resultados processados

### Openai

In [4]:
df_slr2_titulo_resumo_gpt = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_openai_gpt_41_slr2_v2.csv", encoding='utf-8')

df_slr2_titulo_keywords_gpt = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_openai_gpt_41_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_keywords_gpt = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_openai_gpt_41_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_gpt = pd.read_csv("/data/codigos/dados/resultados/resumo_openai_gpt_41_slr2_v2.csv", encoding='utf-8')

In [5]:
# traz todos os resultados ja analisados na fase 1
df_slr2_gpt = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_openai_gpt_41_slr2.csv", encoding='utf-8')
df_slr2_gpt.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,flag_pdfs,tentar_via_vpn,gpt-4.1_IC1_0,gpt-4.1_IC1_1,gpt-4.1_IC1_2,gpt-4.1_IC1_3,gpt-4.1_IC1_4,gpt-4.1_IC2_0,gpt-4.1_IC2_1,gpt-4.1_IC2_2,gpt-4.1_IC2_3,gpt-4.1_IC2_4
0,slr2_1,Redesigning the bartle test of gamer psycholog...,"According to the literature review, to impleme...","Bartle Test, Video Games, Gamification, Motiva...",sucesso,sucesso,não estruturado,sim,NaN,6,6,5,6,6,6,6,6,6,6
1,slr2_2,Towards adaptive gamification: A synthesis of ...,Adaptive gamification is an emerging and fast-...,"Adaptive gamification, structured literature r...",sucesso,sucesso,não estruturado,sim,NaN,6,6,6,6,6,6,6,6,6,6


In [6]:
# só mantem a galera com resultados selecionados no sample
df_slr2_gpt["flag_selecao"] = df_slr2_gpt["ID"].isin(df_slr2_titulo_resumo_gpt["ID"].to_list())
df_slr2_gpt = df_slr2_gpt[df_slr2_gpt["flag_selecao"] == True].copy()
df_slr2_gpt = df_slr2_gpt.drop(columns=["flag_selecao"])
df_slr2_gpt.shape

(44, 19)

### Google

In [7]:
df_slr2_resumo_keywords_google = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_gemini_25_flash_slr2_v2.csv", encoding='utf-8')

df_slr2_titulo_resumo_google = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_gemini_25_flash_slr2_v2.csv", encoding='utf-8')

df_slr2_titulo_keywords_google = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_gemini_25_flash_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_google = pd.read_csv("/data/codigos/dados/resultados/resumo_gemini_25_flash_slr2_v2.csv", encoding='utf-8')


In [8]:
# traz todos os resultados ja analisados na fase 1
df_slr2_google = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_gemini_pro_slr2_v2.csv", encoding='utf-8')
df_slr2_google.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,segunda_coleta,obs,flag_pdfs,tentar_via_vpn,gemini-2.5-flash_IC1_0,gemini-2.5-flash_IC1_1,gemini-2.5-flash_IC1_2,gemini-2.5-flash_IC1_3,gemini-2.5-flash_IC1_4,gemini-2.5-flash_IC2_0,gemini-2.5-flash_IC2_1,gemini-2.5-flash_IC2_2,gemini-2.5-flash_IC2_3,gemini-2.5-flash_IC2_4
0,slr2_1,Redesigning the bartle test of gamer psycholog...,"According to the literature review, to impleme...","Bartle Test, Video Games, Gamification, Motiva...",sucesso,sucesso,não,não estruturado,sim,NaN,6,7,7,7,7,3,3,2,3,3
1,slr2_2,Towards adaptive gamification: A synthesis of ...,Adaptive gamification is an emerging and fast-...,"Adaptive gamification, structured literature r...",sucesso,sucesso,não,não estruturado,sim,NaN,7,7,7,7,7,7,7,7,7,7


In [9]:
# só mantem a galera com resultados selecionados no sample
df_slr2_google["flag_selecao"] = df_slr2_google["ID"].isin(df_slr2_resumo_google["ID"].to_list())
df_slr2_google = df_slr2_google[df_slr2_google["flag_selecao"] == True].copy()
df_slr2_google = df_slr2_google.drop(columns=["flag_selecao"])
df_slr2_google.shape

(50, 20)

### Antropic

In [10]:
df_slr2_titulo_resumo_antropic = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_claude_sonnet_35_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_keywords_antropic = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_claude_sonnet_35_slr2_v2.csv", encoding='utf-8')

df_slr2_titulo_keywords_antropic = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_claude_sonnet_35_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_antropic = pd.read_csv("/data/codigos/dados/resultados/resumo_claude_sonnet_35_slr2_v2.csv", encoding='utf-8')


In [11]:
# traz todos os resultados ja analisados na fase 1
df_slr2_antropic = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_claude_sonnet_35_slr2.csv", encoding='utf-8')
df_slr2_antropic.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,flag_pdfs,tentar_via_vpn,claude-3-5-sonnet-20241022_IC1_0,claude-3-5-sonnet-20241022_IC1_1,claude-3-5-sonnet-20241022_IC1_2,claude-3-5-sonnet-20241022_IC1_3,claude-3-5-sonnet-20241022_IC1_4,claude-3-5-sonnet-20241022_IC2_0,claude-3-5-sonnet-20241022_IC2_1,claude-3-5-sonnet-20241022_IC2_2,claude-3-5-sonnet-20241022_IC2_3,claude-3-5-sonnet-20241022_IC2_4
0,slr2_1,Redesigning the bartle test of gamer psycholog...,"According to the literature review, to impleme...","Bartle Test, Video Games, Gamification, Motiva...",sucesso,sucesso,não estruturado,sim,NaN,7,7,7,7,7,5,5,5,5,5
1,slr2_2,Towards adaptive gamification: A synthesis of ...,Adaptive gamification is an emerging and fast-...,"Adaptive gamification, structured literature r...",sucesso,sucesso,não estruturado,sim,NaN,6,6,6,6,6,6,6,6,6,6


In [12]:
# só mantem a galera com resultados selecionados no sample
df_slr2_antropic["flag_selecao"] = df_slr2_antropic["ID"].isin(df_slr2_resumo_antropic["ID"].to_list())
df_slr2_antropic = df_slr2_antropic[df_slr2_antropic["flag_selecao"] == True].copy()
df_slr2_antropic = df_slr2_antropic.drop(columns=["flag_selecao"])
df_slr2_antropic.shape

(44, 19)

### META

In [13]:
df_slr2_titulo_resumo_meta = pd.read_csv("/data/codigos/dados/resultados/titulo_resumo_llama_4_scout_17B_16E_instruct_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_keywords_meta = pd.read_csv("/data/codigos/dados/resultados/resumo_keywords_llama_4_scout_17B_16E_instruct_slr2_v2.csv", encoding='utf-8')

df_slr2_titulo_keywords_meta = pd.read_csv("/data/codigos/dados/resultados/titulo_keywords_llama_4_scout_17B_16E_instruct_slr2_v2.csv", encoding='utf-8')

df_slr2_resumo_meta = pd.read_csv("/data/codigos/dados/resultados/resumo_llama_4_scout_17B_16E_instruct_slr2_v2.csv", encoding='utf-8')

In [14]:
# traz todos os resultados ja analisados na fase 1
df_slr2_meta = pd.read_csv("/data/codigos/dados/resultados/reproducao_resultados_Llama-4-Scout-17B-16E-Instruct_slr2_v2.csv", encoding='utf-8')
df_slr2_meta.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,segunda_coleta,obs,flag_pdfs,tentar_via_vpn,Llama-4-Scout-17B-16E-Instruct_IC1_0,Llama-4-Scout-17B-16E-Instruct_IC1_1,Llama-4-Scout-17B-16E-Instruct_IC1_2,Llama-4-Scout-17B-16E-Instruct_IC1_3,Llama-4-Scout-17B-16E-Instruct_IC1_4,Llama-4-Scout-17B-16E-Instruct_IC2_0,Llama-4-Scout-17B-16E-Instruct_IC2_1,Llama-4-Scout-17B-16E-Instruct_IC2_2,Llama-4-Scout-17B-16E-Instruct_IC2_3,Llama-4-Scout-17B-16E-Instruct_IC2_4
0,slr2_1,Redesigning the bartle test of gamer psycholog...,"According to the literature review, to impleme...","Bartle Test, Video Games, Gamification, Motiva...",sucesso,sucesso,não,não estruturado,sim,NaN,2,2,2,2,2,2,2,2,2,2
1,slr2_2,Towards adaptive gamification: A synthesis of ...,Adaptive gamification is an emerging and fast-...,"Adaptive gamification, structured literature r...",sucesso,sucesso,não,não estruturado,sim,NaN,7,7,7,7,7,5,5,5,5,5


In [15]:
# só mantem a galera com resultados selecionados no sample
df_slr2_meta["flag_selecao"] = df_slr2_meta["ID"].isin(df_slr2_resumo_meta["ID"].to_list())
df_slr2_meta = df_slr2_meta[df_slr2_meta["flag_selecao"] == True].copy()
df_slr2_meta = df_slr2_meta.drop(columns=["flag_selecao"])
df_slr2_meta.shape

(50, 20)

### joins

In [16]:
# vamos trazer os dados disponibilizados pelos autores para comparação de performance
df_slr2_autores = pd.read_excel("/data/codigos/dados/slr2-results-keys.xlsx",engine='openpyxl')
df_slr2_autores.head()

,ID,Title,IC1,IC2,Benchmark,Execution time (s),Total Tokens
0,slr2_1,Redesigning the bartle test of gamer psycholog...,6,6,E,"1,7697742920136",777
1,slr2_2,Towards adaptive gamification: A synthesis of ...,5,5,I,"1,26788704199134",889
2,slr2_3,Field guide to gamification: Game components a...,7,7,I,"1,3752188339713",1233
3,slr2_4,Gamification of eHealth interventions to incre...,1,2,I,"1,54043745799572",1047
4,slr2_5,The relationship between gender and game dynam...,6,6,I,1.338598,1175


In [17]:
df_slr2_titulo_resumo_gpt = pd.merge(left=df_slr2_titulo_resumo_gpt, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_resumo_gpt.shape


(50, 16)

In [18]:
df_slr2_titulo_keywords_gpt = pd.merge(left=df_slr2_titulo_keywords_gpt, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_keywords_gpt.shape

(50, 16)

In [19]:
df_slr2_resumo_keywords_gpt = pd.merge(left=df_slr2_resumo_keywords_gpt, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_keywords_gpt.shape

(50, 16)

In [20]:
df_slr2_resumo_gpt = pd.merge(left=df_slr2_resumo_gpt, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_gpt.shape

(50, 15)

In [21]:
df_slr2_gpt = pd.merge(left=df_slr2_gpt, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_gpt.shape

(44, 22)

In [22]:
df_slr2_resumo_keywords_google = pd.merge(left=df_slr2_resumo_keywords_google, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_keywords_google.shape

(50, 16)

In [23]:
df_slr2_titulo_resumo_google = pd.merge(left=df_slr2_titulo_resumo_google, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_resumo_google.shape

(50, 16)

In [24]:
df_slr2_titulo_keywords_google = pd.merge(left=df_slr2_titulo_keywords_google, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_keywords_google.shape

(50, 16)

In [25]:
df_slr2_resumo_google = pd.merge(left=df_slr2_resumo_google, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_google.shape

(50, 15)

In [26]:
df_slr2_google = pd.merge(left=df_slr2_google, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_google.shape

(50, 23)

In [27]:
df_slr2_titulo_resumo_antropic = pd.merge(left=df_slr2_titulo_resumo_antropic, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_resumo_antropic.shape

(50, 16)

In [28]:
df_slr2_resumo_keywords_antropic = pd.merge(left=df_slr2_resumo_keywords_antropic, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_keywords_antropic.shape

(50, 16)

In [29]:
df_slr2_titulo_keywords_antropic = pd.merge(left=df_slr2_titulo_keywords_antropic, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_keywords_antropic.shape

(50, 16)

In [30]:
df_slr2_resumo_antropic = pd.merge(left=df_slr2_resumo_antropic, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_antropic.shape

(50, 15)

In [31]:
df_slr2_antropic = pd.merge(left=df_slr2_antropic, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_antropic.shape

(44, 22)

In [32]:
df_slr2_titulo_resumo_meta = pd.merge(left=df_slr2_titulo_resumo_meta, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_resumo_meta.shape

(50, 16)

In [33]:
df_slr2_resumo_keywords_meta = pd.merge(left=df_slr2_resumo_keywords_meta, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_keywords_meta.shape

(50, 16)

In [34]:
df_slr2_titulo_keywords_meta = pd.merge(left=df_slr2_titulo_keywords_meta, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_titulo_keywords_meta.shape

(50, 16)

In [35]:
df_slr2_resumo_meta = pd.merge(left=df_slr2_resumo_meta, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_resumo_meta.shape

(50, 15)

In [36]:
df_slr2_meta = pd.merge(left=df_slr2_meta, right=df_slr2_autores[["ID","IC1","IC2","Benchmark"]],
                          left_on='ID', right_on='ID',how='left')
df_slr2_meta.shape

(50, 23)

### Geração dos resultados das llms e padronizacao dos campos

In [37]:
# criando a regra do corte em 5
def convert_benchmark(df,col1):
    df["result_bench"] = np.where(df[col1].str.lower()=="i",1,0)
    return df

In [38]:
dicto_resultados = {"df_slr2_titulo_resumo_gpt":df_slr2_titulo_resumo_gpt,
                    "df_slr2_titulo_keywords_gpt":df_slr2_titulo_keywords_gpt,
                    "df_slr2_resumo_keywords_gpt":df_slr2_resumo_keywords_gpt,
                    "df_slr2_resumo_gpt":df_slr2_resumo_gpt,
                    "df_slr2_gpt":df_slr2_gpt,
                    "df_slr2_resumo_keywords_google":df_slr2_resumo_keywords_google,
                    "df_slr2_titulo_resumo_google":df_slr2_titulo_resumo_google,
                    "df_slr2_titulo_keywords_google":df_slr2_titulo_keywords_google,
                    "df_slr2_resumo_google":df_slr2_resumo_google,
                    "df_slr2_google":df_slr2_google,
                    "df_slr2_titulo_resumo_antropic":df_slr2_titulo_resumo_antropic,
                    "df_slr2_resumo_keywords_antropic":df_slr2_resumo_keywords_antropic,
                    "df_slr2_titulo_keywords_antropic":df_slr2_titulo_keywords_antropic,
                    "df_slr2_resumo_antropic":df_slr2_resumo_antropic,
                    "df_slr2_antropic":df_slr2_antropic,
                    "df_slr2_titulo_resumo_meta":df_slr2_titulo_resumo_meta,
                    "df_slr2_resumo_keywords_meta":df_slr2_resumo_keywords_meta,
                    "df_slr2_titulo_keywords_meta":df_slr2_titulo_keywords_meta,
                    "df_slr2_resumo_meta":df_slr2_resumo_meta,
                    "df_slr2_meta":df_slr2_meta}

dicto_modelos = {"df_slr2_titulo_resumo_gpt":"gpt-4.1",
                    "df_slr2_titulo_keywords_gpt":"gpt-4.1",
                    "df_slr2_resumo_keywords_gpt":"gpt-4.1",
                    "df_slr2_resumo_gpt":"gpt-4.1",
                    "df_slr2_gpt":"gpt-4.1",
                    "df_slr2_resumo_keywords_google":"gemini-2.5-flash",
                    "df_slr2_titulo_resumo_google":"gemini-2.5-flash",
                    "df_slr2_titulo_keywords_google":"gemini-2.5-flash",
                    "df_slr2_resumo_google":"gemini-2.5-flash",
                    "df_slr2_google":"gemini-2.5-flash",
                    "df_slr2_titulo_resumo_antropic":"claude-3-5-sonnet-20241022",
                    "df_slr2_resumo_keywords_antropic":"claude-3-5-sonnet-20241022",
                    "df_slr2_titulo_keywords_antropic":"claude-3-5-sonnet-20241022",
                    "df_slr2_resumo_antropic":"claude-3-5-sonnet-20241022",
                    "df_slr2_antropic":"claude-3-5-sonnet-20241022",
                "df_slr2_titulo_resumo_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr2_resumo_keywords_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr2_titulo_keywords_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr2_resumo_meta":"Llama-4-Scout-17B-16E-Instruct",
                    "df_slr2_meta":"Llama-4-Scout-17B-16E-Instruct"}

dicto_selecao = {"df_slr2_titulo_resumo_gpt":"title+abstract",
                    "df_slr2_titulo_keywords_gpt":"title+keywords",
                    "df_slr2_resumo_keywords_gpt":"abstract+keywords",
                    "df_slr2_resumo_gpt":"abstract",
                    "df_slr2_gpt":"title+abstract+keywords",
                    "df_slr2_resumo_keywords_google":"abstract+keywords",
                    "df_slr2_titulo_resumo_google":"title+abstract",
                    "df_slr2_titulo_keywords_google":"title+keywords",
                    "df_slr2_resumo_google":"abstract",
                    "df_slr2_google":"title+abstract+keywords",
                    "df_slr2_titulo_resumo_antropic":"title+abstract",
                    "df_slr2_resumo_keywords_antropic":"abstract+keywords",
                    "df_slr2_titulo_keywords_antropic":"title+keywords",
                    "df_slr2_resumo_antropic":"abstract",
                    "df_slr2_antropic":"title+abstract+keywords",
                "df_slr2_titulo_resumo_meta":"title+abstract",
                    "df_slr2_resumo_keywords_meta":"abstract+keywords",
                    "df_slr2_titulo_keywords_meta":"title+keywords",
                    "df_slr2_resumo_meta":"abstract",
                    "df_slr2_meta":"title+abstract+keywords"}


In [39]:
for key in dicto_resultados:
    dicto_resultados[key] = convert_benchmark(dicto_resultados[key], "Benchmark")

In [40]:
df_slr2_antropic.head(2)

,ID,title,abstract,keywords,sucesso_fracasso_resumo,sucesso_fracasso_palavras,obs,flag_pdfs,tentar_via_vpn,claude-3-5-sonnet-20241022_IC1_0,claude-3-5-sonnet-20241022_IC1_1,claude-3-5-sonnet-20241022_IC1_2,claude-3-5-sonnet-20241022_IC1_3,claude-3-5-sonnet-20241022_IC1_4,claude-3-5-sonnet-20241022_IC2_0,claude-3-5-sonnet-20241022_IC2_1,claude-3-5-sonnet-20241022_IC2_2,claude-3-5-sonnet-20241022_IC2_3,claude-3-5-sonnet-20241022_IC2_4,IC1,IC2,Benchmark,result_bench
0,slr2_1,Redesigning the bartle test of gamer psycholog...,"According to the literature review, to impleme...","Bartle Test, Video Games, Gamification, Motiva...",sucesso,sucesso,não estruturado,sim,NaN,7,7,7,7,7,5,5,5,5,5,6,6,E,0
1,slr2_10,An investigation of the impact of gamification...,Gamification is becoming a popular classroom i...,"gamification, meaningful gamification, novice ...",sucesso,sucesso,não estruturado,sim,NaN,7,7,7,7,7,6,6,6,6,6,6,6,I,1


In [41]:
def create_result_llm_classic(df,col1,col2,limit1,limit2,col_name):
    """
    Create flag with result for llms
    """
    df["aux_ic1"] = np.where(df[col1]>=limit1,1,0)
    df["aux_ic2"] = np.where(df[col2]>=limit2,1,0)
    df[col_name] = np.where(((df["aux_ic1"]==1) & (df["aux_ic2"]==1)),1,0)
    return df.drop(columns=["aux_ic1","aux_ic2"])

In [42]:
# Cria a resposta final para cada iteração
def calculate_select_result_llm(n_interactions, model_gpt, df):
    name_ic1 = "IC1"
    name_ic2 = "IC2"
    for i in range(n_interactions):
        coluna_aux1 = model_gpt+"_"+name_ic1+"_"+str(i)
        coluna_aux2 = model_gpt+"_"+name_ic2+"_"+str(i)
        col_name = "result_llm"+"_"+str(i)
        df = create_result_llm_classic(df=df,
                                        col1=coluna_aux1,
                                        col2=coluna_aux2,
                                        limit1=5,
                                        limit2=5,
                                        col_name=col_name)
    return df

In [43]:
n_interactions = 5

In [44]:
for key in dicto_resultados:
    print(key)
    dicto_resultados[key] = calculate_select_result_llm(n_interactions=n_interactions, 
                                                        model_gpt=dicto_modelos[key],
                                                        df=dicto_resultados[key])

df_slr2_titulo_resumo_gpt
df_slr2_titulo_keywords_gpt
df_slr2_resumo_keywords_gpt
df_slr2_resumo_gpt
df_slr2_gpt
df_slr2_resumo_keywords_google
df_slr2_titulo_resumo_google
df_slr2_titulo_keywords_google
df_slr2_resumo_google
df_slr2_google
df_slr2_titulo_resumo_antropic
df_slr2_resumo_keywords_antropic
df_slr2_titulo_keywords_antropic
df_slr2_resumo_antropic
df_slr2_antropic
df_slr2_titulo_resumo_meta
df_slr2_resumo_keywords_meta
df_slr2_titulo_keywords_meta
df_slr2_resumo_meta
df_slr2_meta


In [45]:
dicto_resultados["df_slr2_titulo_resumo_gpt"].head(2)

,ID,title,abstract,gpt-4.1_IC1_0,gpt-4.1_IC1_1,gpt-4.1_IC1_2,gpt-4.1_IC1_3,gpt-4.1_IC1_4,gpt-4.1_IC2_0,gpt-4.1_IC2_1,gpt-4.1_IC2_2,gpt-4.1_IC2_3,gpt-4.1_IC2_4,IC1,IC2,Benchmark,result_bench,result_llm_0,result_llm_1,result_llm_2,result_llm_3,result_llm_4
0,slr2_81,A Gamification Model For E-Learning Platforms,The use of gamification in education has been ...,5,5,5,5,6,5,5,5,5,5,2,2,I,1,1,1,1,1,1
1,slr2_318,Preschoolers' Understanding of a Teachable Age...,This study investigated how preschool children...,4,4,4,4,4,3,3,3,3,3,2,2,E,0,0,0,0,0,0


### Ajustando os dataframes

In [46]:

for key in dicto_resultados:
    print(key)
    old=dicto_modelos[key]
    new="llm"
    dicto_resultados[key]["model_id"]  = dicto_modelos[key]
    dicto_resultados[key]["set_features"]  = dicto_selecao[key]
    dicto_resultados[key]["slr"]  = "slr2"
    dicto_resultados[key].columns = dicto_resultados[key].columns.str.replace(old, new, regex=True)

df_slr2_titulo_resumo_gpt
df_slr2_titulo_keywords_gpt
df_slr2_resumo_keywords_gpt
df_slr2_resumo_gpt
df_slr2_gpt
df_slr2_resumo_keywords_google
df_slr2_titulo_resumo_google
df_slr2_titulo_keywords_google
df_slr2_resumo_google
df_slr2_google
df_slr2_titulo_resumo_antropic
df_slr2_resumo_keywords_antropic
df_slr2_titulo_keywords_antropic
df_slr2_resumo_antropic
df_slr2_antropic
df_slr2_titulo_resumo_meta
df_slr2_resumo_keywords_meta
df_slr2_titulo_keywords_meta
df_slr2_resumo_meta
df_slr2_meta


In [47]:
dicto_resultados["df_slr2_titulo_resumo_gpt"].head(2)

,ID,title,abstract,llm_IC1_0,llm_IC1_1,llm_IC1_2,llm_IC1_3,llm_IC1_4,llm_IC2_0,llm_IC2_1,llm_IC2_2,llm_IC2_3,llm_IC2_4,IC1,IC2,Benchmark,result_bench,result_llm_0,result_llm_1,result_llm_2,result_llm_3,result_llm_4,model_id,set_features,slr
0,slr2_81,A Gamification Model For E-Learning Platforms,The use of gamification in education has been ...,5,5,5,5,6,5,5,5,5,5,2,2,I,1,1,1,1,1,1,gpt-4.1,title+abstract,slr2
1,slr2_318,Preschoolers' Understanding of a Teachable Age...,This study investigated how preschool children...,4,4,4,4,4,3,3,3,3,3,2,2,E,0,0,0,0,0,0,gpt-4.1,title+abstract,slr2


In [48]:
lista_colunas_indentificadores = ["ID","slr","model_id","set_features"]
lista_colunas_ics = ["IC1","IC2","llm_IC1_0","llm_IC2_0","llm_IC1_1","llm_IC2_1","llm_IC1_2",
                     "llm_IC2_2","llm_IC1_3","llm_IC2_3","llm_IC1_4","llm_IC2_4"]
lista_colunas_resultados = ["result_bench","result_llm_0","result_llm_1","result_llm_2","result_llm_3","result_llm_4"]

In [49]:
# empilha os resultados em uma unico dataframe
df_resultados_empilhados = pd.DataFrame()
for key in dicto_resultados:
    df_temp = dicto_resultados[key][lista_colunas_indentificadores + lista_colunas_ics + lista_colunas_resultados].copy()
    df_resultados_empilhados = pd.concat([df_resultados_empilhados, df_temp], axis=0, ignore_index=True)

In [50]:
df_resultados_empilhados.head()

,ID,slr,model_id,set_features,IC1,IC2,llm_IC1_0,llm_IC2_0,llm_IC1_1,llm_IC2_1,llm_IC1_2,llm_IC2_2,llm_IC1_3,llm_IC2_3,llm_IC1_4,llm_IC2_4,result_bench,result_llm_0,result_llm_1,result_llm_2,result_llm_3,result_llm_4
0,slr2_81,slr2,gpt-4.1,title+abstract,2,2,5,5,5,5,5,5,5,5,6,5,1,1,1,1,1,1
1,slr2_318,slr2,gpt-4.1,title+abstract,2,2,4,3,4,3,4,3,4,3,4,3,0,0,0,0,0,0
2,slr2_285,slr2,gpt-4.1,title+abstract,5,2,3,2,3,2,3,2,3,2,3,2,0,0,0,0,0,0
3,slr2_58,slr2,gpt-4.1,title+abstract,6,6,6,5,6,5,6,5,6,5,6,5,1,1,1,1,1,1
4,slr2_444,slr2,gpt-4.1,title+abstract,4,2,4,1,1,1,4,1,1,1,4,2,0,0,0,0,0,0


In [51]:
df_resultados_empilhados.shape

(988, 22)

In [52]:
df_resultados_empilhados["model_id"].unique()

array(['gpt-4.1', 'gemini-2.5-flash', 'claude-3-5-sonnet-20241022',
       'Llama-4-Scout-17B-16E-Instruct'], dtype=object)

In [53]:
# vendo se os indentificadores juntos formam uma chave unica
df_resultados_empilhados["chave_unica"] = df_resultados_empilhados["ID"].astype(str) + "_" + df_resultados_empilhados["slr"].astype(str) + "_" + df_resultados_empilhados["model_id"].astype(str) + "_" + df_resultados_empilhados["set_features"].astype(str) 
df_resultados_empilhados["chave_unica"].nunique() == df_resultados_empilhados.shape[0]


True

In [54]:
# deleta chave unica
df_resultados_empilhados = df_resultados_empilhados.drop(columns=["chave_unica"])
df_resultados_empilhados.shape

(988, 22)

In [55]:
# grava os resultados em excel e csv
df_resultados_empilhados.to_excel("/data/codigos/dados/resultados/resultados_empilhados_llms_slr2_prep_GLMM_v2.xlsx", index=False)
df_resultados_empilhados.to_csv("/data/codigos/dados/resultados/resultados_empilhados_llms_slr2_prep_GLMM_v2.csv", index=False, encoding='utf-8')
